# 3-3. 쿼리 변환 (Query Transformation)

## 학습 목표
- 사용자의 원본 질문을 LLM으로 **변형·확장**해 검색 성능을 높이는 주요 기법 5가지를 구현할 수 있다.
- 각 기법이 잘 맞는 질문 유형과 실패 모드를 설명할 수 있다.

## 핵심 키워드
HyDE · Multi-Query · Step-Back · RAG-Fusion · Query Decomposition


In [ ]:
# 공통 세팅: common 헬퍼를 사용하기 위해 repo 루트를 sys.path에 추가
import sys, os
sys.path.insert(0, '../..')

from common import get_chat_model, get_embeddings, provider_badge
from common.ko_tokenizer import tokenize_ko
from common.loaders import load_any

print(provider_badge())


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

docs = load_any('../../data/corpus_ko.txt')
chunks = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40).split_documents(docs)
vectordb = FAISS.from_documents(chunks, get_embeddings())
retriever = vectordb.as_retriever(search_kwargs={'k': 3})
llm = get_chat_model(temperature=0.0)
print(f'청크 수: {len(chunks)}')


## 1. HyDE (Hypothetical Document Embedding)

> "질문"의 임베딩과 "답변"의 임베딩은 벡터 공간에서 꽤 떨어져 있다. LLM으로 **가상의 답변 문서**를 먼저 생성한 뒤 그것을 임베딩해 검색하면 recall이 올라간다.

```
질문 ─▶ LLM ─▶ 가상 답변 ─▶ 임베딩 ─▶ 벡터 검색 ─▶ 실제 문서
```


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

hyde_prompt = ChatPromptTemplate.from_template(
    '아래 질문에 대해 간결한 가상의 답변 한 단락을 작성하라. 사실이 아니어도 된다.\n\n질문: {q}\n답변:'
)
hyde_chain = hyde_prompt | llm | StrOutputParser()

def hyde_retrieve(q: str, k: int = 3):
    fake = hyde_chain.invoke({'q': q})
    # 가상 답변을 쿼리로 사용해 벡터 검색
    return fake, vectordb.similarity_search(fake, k=k)

fake, docs_hyde = hyde_retrieve('전자금융거래의 정의는?')
print('[생성된 가상 답변]\n', fake)
print('\n[검색된 문서 top-3]')
for d in docs_hyde:
    print('-', d.page_content[:80])


**적합 상황**: 질문이 짧거나 추상적이어서 실제 문서 문장과 표면 유사도가 낮은 경우.
**실패 모드**: LLM이 환각을 크게 일으켜 엉뚱한 방향의 가상 답변을 만들면 오히려 검색이 악화된다.


## 2. Multi-Query Retriever

하나의 질문을 **여러 관점·표현**으로 다시 쓰고, 각 쿼리로 검색한 결과의 합집합을 돌려준다.

In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever
import logging
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

mq_retriever = MultiQueryRetriever.from_llm(retriever=retriever, llm=llm)
result = mq_retriever.invoke('망분리는 무엇인가?')
print(f'총 {len(result)}개 문서')
for d in result:
    print('-', d.page_content[:70])


## 3. Step-Back Prompting

구체적인 질문을 **더 일반적(상위 개념)인 질문**으로 변환해 배경 지식을 먼저 끌어온 뒤, 원문 질문에 답한다. 추론형 질문에서 특히 효과적이다.

```
원문: "Llama3 8B 모델의 KV 캐시 크기는 어떻게 계산하나요?"
 ▼
step-back: "트랜스포머 모델의 KV 캐시는 일반적으로 어떻게 계산되는가?"
```


In [ ]:
step_back_prompt = ChatPromptTemplate.from_template(
    '다음 질문을 더 일반적이고 추상적인 상위 개념 질문 1개로 다시 써라.\n'
    '상위 질문만 한 줄로 출력하라.\n\n원문: {q}'
)
step_back_chain = step_back_prompt | llm | StrOutputParser()

q = '전자금융감독규정에서 전산실 운영은 어떤 망에서 해야 하는가?'
sb = step_back_chain.invoke({'q': q})
print('원문     :', q)
print('step-back:', sb)
print('\n[step-back으로 검색]')
for d in retriever.invoke(sb):
    print('-', d.page_content[:70])


## 4. RAG-Fusion

Multi-Query와 RRF의 결합: N개 쿼리 변형을 생성 → 각각 검색 → **RRF로 재정렬**해 top-K 반환.

In [ ]:
from langchain_core.runnables import RunnablePassthrough

multi_q_prompt = ChatPromptTemplate.from_template(
    '아래 질문을 의미는 같되 표현이 다른 4개 질문으로 다시 써라. 번호 없이 줄바꿈으로 구분:\n{q}'
)
gen_queries = multi_q_prompt | llm | StrOutputParser() | (lambda x: [s for s in x.splitlines() if s.strip()])

def rrf_docs(lists, k=60):
    scores = {}
    doc_map = {}
    for lst in lists:
        for rank, d in enumerate(lst, start=1):
            key = d.page_content
            scores[key] = scores.get(key, 0.0) + 1.0 / (k + rank)
            doc_map[key] = d
    sorted_keys = sorted(scores, key=scores.get, reverse=True)
    return [doc_map[k] for k in sorted_keys]

def rag_fusion(q: str, top=4):
    queries = [q] + gen_queries.invoke({'q': q})
    ranked_lists = [retriever.invoke(qq) for qq in queries]
    merged = rrf_docs(ranked_lists)
    return queries, merged[:top]

qs, docs_fused = rag_fusion('신용정보 처리위탁 시 금융회사의 의무는?')
print('[생성된 쿼리 변형]')
for q in qs:
    print(' •', q)
print('\n[RRF 병합 결과]')
for d in docs_fused:
    print('-', d.page_content[:80])


## 5. Query Decomposition (쿼리 분해)

복합 질문을 원자적(atomic) 하위 질문으로 쪼갠 뒤 각각 검색·답변하고, 최종적으로 합성한다.

> 예: "삼성전자의 자회사는 어디고 그 사업분야는?" → (a) "삼성전자의 자회사는?" (b) "{자회사}의 사업분야는?"


In [ ]:
decompose_prompt = ChatPromptTemplate.from_template(
    '사용자 질문을 답을 위해 필요한 원자적 하위질문으로 분해하라. 각 줄에 한 개씩, 번호 없이 출력:\n\n{q}'
)
decompose_chain = decompose_prompt | llm | StrOutputParser() | (lambda x: [s.strip('- ').strip() for s in x.splitlines() if s.strip()])

q = '망분리 원칙의 법적 근거와 업무망·인터넷망의 차이를 설명해줘.'
sub_qs = decompose_chain.invoke({'q': q})
print('[분해된 하위질문]')
for s in sub_qs:
    print(' -', s)

# 하위질문별 검색
for s in sub_qs:
    print(f'\n▶ {s}')
    for d in retriever.invoke(s)[:2]:
        print('  •', d.page_content[:70])


## 6. 기법 선택 가이드

| 질문 유형 | 추천 기법 |
|-----------|-----------|
| 짧고 키워드 부족 | HyDE, Multi-Query |
| 복합/다중 엔터티 | Query Decomposition |
| 추론·배경 지식 필요 | Step-Back |
| 일반 RAG 품질 향상 | RAG-Fusion |

> ⚠️ 모든 기법은 LLM 호출이 1~N회 추가된다. **지연과 비용을 반드시 RAGAS 점수와 함께 측정**해 trade-off를 정량화할 것.
